# 01 — Dataset audit (v2)

Audit degli split v2: composizione, lunghezze, distanze tipo B, bilanciamento label e MCQ, duplicati residui. Legge SOLO `data/processed/` (validato dal contratto v2) e il manifest.

Output: figure in `results/figures/`, tabella riassuntiva in `results/tables/`.

In [ ]:
# Provenance header: every figure must be traceable to the state that made it.
import json, subprocess, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

def sh(*cmd):
    try:
        return subprocess.check_output(cmd, cwd=ROOT, text=True).strip()
    except Exception as e:
        return f"<unavailable: {e}>"

GIT_COMMIT = sh("git", "rev-parse", "--short", "HEAD")
GIT_DIRTY = bool(sh("git", "status", "--porcelain"))
MANIFEST_PATH = ROOT / "data/processed/manifest.json"
MANIFEST = json.loads(MANIFEST_PATH.read_text()) if MANIFEST_PATH.exists() else None
print(f"commit={GIT_COMMIT} dirty={GIT_DIRTY}")
print(f"manifest: {'present, created ' + MANIFEST['created_at'] if MANIFEST else 'ABSENT'}")


In [ ]:
from src.data_contract import load_jsonl_validated
splits = {}
for name in ("train", "eval", "test", "probe"):
    path = ROOT / f"data/processed/{name}.jsonl"
    splits[name] = load_jsonl_validated(path) if path.exists() else []
    print(f"{name}: {len(splits[name])} righe validate")


In [ ]:
# Composizione per tipo / sorgente / label_kind / annotazione MCQ
import pandas as pd
rows = []
for name, exs in splits.items():
    for ex in exs:
        m = ex["meta"]
        rows.append({"split": name, "type": ex["type"], "source": m["source"],
                     "label_kind": m.get("label_kind") or "-",
                     "has_mcq": bool(m.get("options")),
                     "distance": m.get("distance_target_tokens"),
                     "ctx_words": len(ex["context"].split()),
                     "tgt_words": len(ex["target"].split()),
                     "fill_words": len(ex["filler"].split())})
df = pd.DataFrame(rows)
summary = df.groupby("split").agg(
    n=("type", "size"), mcq=("has_mcq", "sum"),
    ctx_words_median=("ctx_words", "median"),
    tgt_words_median=("tgt_words", "median"))
display(summary)
(ROOT / "results/tables").mkdir(parents=True, exist_ok=True)
summary.to_csv(ROOT / "results/tables/dataset_summary.csv")
display(pd.crosstab(df["split"], df["type"]))
display(pd.crosstab(df["split"], df["source"]))


In [ ]:
# Lunghezze contesto/target per split — istogrammi a serie singola
import matplotlib.pyplot as plt
HUE = "#2a78d6"
fig, axes = plt.subplots(2, 4, figsize=(14, 5), sharey="row")
for col, name in enumerate(("train", "eval", "test", "probe")):
    sub = df[df["split"] == name]
    for row, field in enumerate(("ctx_words", "tgt_words")):
        ax = axes[row][col]
        ax.hist(sub[field], bins=30, color=HUE, edgecolor="white", linewidth=0.4)
        ax.set_title(f"{name} — {field}", fontsize=9)
        ax.spines[["top", "right"]].set_visible(False)
        ax.grid(axis="y", alpha=0.25)
fig.suptitle("Distribuzione lunghezze (parole)", y=1.02)
fig.tight_layout()
(ROOT / "results/figures").mkdir(parents=True, exist_ok=True)
fig.savefig(ROOT / "results/figures/dataset_lengths.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Tipo B: distanza target dichiarata (token) — la distanza EFFETTIVA sul
# tokenizer di training sara' misurata dalla pipeline (campo distance_actual_tokens)
b = df[(df["type"] == "B") & df["distance"].notna()]
if len(b):
    import matplotlib.pyplot as plt
    counts = b.groupby(["split", "distance"]).size().unstack(fill_value=0)
    display(counts)
    ax = b[b["split"] == "train"]["distance"].plot.hist(
        bins=30, color="#2a78d6", edgecolor="white", figsize=(7, 3))
    ax.set_xlabel("distance_target_tokens (train, tipo B)")
    ax.spines[["top", "right"]].set_visible(False)
    plt.show()


In [ ]:
# Bilanciamento label (probe) e posizione risposta MCQ (test)
import pandas as pd
probe_labels = pd.Series([ex["meta"].get("label") for ex in splits["probe"]])
print("probe label balance:")
print(probe_labels.value_counts())
answer_pos = pd.Series([ex["meta"].get("answer_idx") for ex in splits["test"]
                        if ex["meta"].get("options") is not None])
print("\ntest MCQ answer_idx distribution (bias posizionale se sbilanciata):")
print(answer_pos.value_counts().sort_index())


In [ ]:
# Duplicati residui per content_id (attesi: 0 dentro ogni split; probe
# condivide contenuti con eval+test by design)
for name, exs in splits.items():
    ids = [ex["content_id"] for ex in exs]
    print(f"{name}: {len(ids) - len(set(ids))} duplicati interni")


In [ ]:
# Gallery: 3 esempi casuali dal test (seed fisso per riproducibilita')
import random
rng = random.Random(42)
for ex in rng.sample(splits["test"], 3):
    m = ex["meta"]
    print("=" * 78)
    print(f"[{ex['type']} | {m['source']} | id {ex['example_id']}]")
    print("CONTEXT:", ex["context"][:280], "..." if len(ex["context"]) > 280 else "")
    print("TARGET :", ex["target"][:200])
    if m.get("question"):
        print("MCQ    :", m["question"], "| answer:", m["options"][m["answer_idx"]])
